In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module for MongoDB interactions
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Connect to database via CRUD Module
# Credentials are handled within the AnimalShelter class
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

app.layout = html.Div([

    # HEADER (Logo + Title + Name)
    html.Div([
        html.A(
            html.Img(
                src='assets/Grazioso Salvare Logo.png',
                style={'height': '280px', 'marginBottom': '1px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.H2('CS-340 Dashboard', 
        style={'color': '#D11C3E', 'marginBottom': '1px'}),
        html.H4('Developed by Kevin David')
    ], style={'textAlign': 'center', 'marginTop': '1px',
        'marginBottom': '20px'}),

    html.Hr(),

    # FILTER PLACEHOLDER
    html.Div([
        html.H4("Filter Options",
        style={'color': '#D11C3E', 'marginBottom': '2px'}),
    
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'All', 'value': 'ALL'},
            {'label': 'Water Rescue', 'value': 'WATER'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'MOUNTAIN'},
            {'label': 'Disaster or Individual Tracking', 'value': 'DISASTER'}
        ],
        value='ALL',
        labelStyle={'display': 'inline-block', 'margin': '10px'}
    )
], style={'textAlign': 'center'}),

    html.Hr(),

    # DATA TABLE
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),
        page_size=10,
        row_selectable='single',
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'backgroundColor': '#f0f0f0',
            'color': 'black',
            'minWidth': '100px',
            'width': '100px',
            'maxWidth': '180px',
        },
        style_header={
            'backgroundColor': '#D11C3E',
            'color': 'white',
            'fontWeight': 'bold'
        }
    ),

    html.Br(),
    html.Hr(),

    # GRAPH + MAP SIDE BY SIDE
    html.Div(
        className='row',
        style={'display': 'flex', 'gap': '20px', 'padding': '20px;'},
        children=[

            html.Div(
                id='graph-id',
                style={'flex': '1'}
                #className='col s12 m6',
            ),

            html.Div(
                id='map-id',
                style={'flex': '1'}
                #className='col s12 m6',
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################
    
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    # Default (All)
    if filter_type == 'ALL':
        return df.to_dict('records')

    # Water Rescue
    elif filter_type == 'WATER':
        query = {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    # Mountain Rescue
    elif filter_type == 'MOUNTAIN':
        query = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    # Disaster
    elif filter_type == 'DISASTER':
        query = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    # Run query
    dff = pd.DataFrame.from_records(db.read(query))

    # Remove ObjectId column
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')
# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    # If no data, return empty
    if viewData is None or len(viewData) == 0:
        return []

    # Convert to dataframe
    dff = pd.DataFrame.from_dict(viewData)

    # Create pie chart by breed
    fig = px.pie(
        dff,
        names='breed',
        title='Distribution of Breeds'
    )

    return [
        dcc.Graph(figure=fig)
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    # Prevent crash when nothing is selected on initial load
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):  

    # If no data, return empty
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # If nothing selected, default to first row
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Austin TX center
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[dff.iloc[row, 13], dff.iloc[row, 14]],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(f"{dff.iloc[row, 9]} - {dff.iloc[row, 4]}"),
                        dl.Popup([
                            html.H3(dff.iloc[row, 9]),
                            html.P(dff.iloc[row, 4], style={'fontsize': '14px', 'color': 'gray'})
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://tunnelnatasha-sambaquality-3000.codio.io/proxy/8050/
